In [16]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import einops as eops


In [17]:
class NextStateModel():
    def __init__(self, _nState :int, _nAction :int):
        self._nState = _nState
        self._nAction = _nAction
        self.A = np.eye(_nState)
        self.B = np.zeros(shape=(_nState, _nAction))

    def train(self, States :np.ndarray, Actions :np.ndarray, lr :float=1e-2, epochs :int=1000):
        if States.ndim != 2 or Actions.ndim != 2:
            print("Dimension mismatch")
            return
            
        _nState, _nSample = States.shape
        _nAction, _ = Actions.shape


        if _nState!=self._nState or _nAction != self._nAction:
            print(f"Req:({self._nState},{self._nAction}) provided:({_nState},{_nAction})")
            return


        A=self.A.copy()
        B=self.B.copy()

        for j in range(epochs):
            prev_state = States[:, 0:1]
            action = Actions[:, 0:1]

            dL_dA = 0
            dL_dB = 0
            err_total = 0
            for i in range(1, _nSample):
                curr_state = States[:, i:i+1]
    
                pred = A@prev_state + B@action
                err = curr_state - pred

                err_total += err.mean()*100
                
                dL_dA -= (2*err) @ prev_state.T
                dL_dB -= (2*err) @ action.T

                prev_state = curr_state
                action = Actions[:, i:i+1]
                
            dL_dA /= _nSample
            dL_dB /= _nSample

            err_total /= 699
            print(f"Err standing at {abs(err_total)}% at {j}th epochs")

            A -= lr*dL_dA
            B -= lr*dL_dB


        self.A = A
        self.B = B

    def __predict__(self, state :np.ndarray, action :np.ndarray) -> np.ndarray:
        if state.shape[0] != self._nState or action.shape[0] != self._nAction:
            print("Dim mismatch")
            return None

        return (self.A@state + self.B@action)

In [ ]:
data = pd.read_csv("inverted_pendulum_state.csv")
dropped_columns = ['Pendulum_Angle_Deg', 'Action_Type', 'Gravity_Setting', 'Timestamp(s)', 'Force_Applied_N']
data = data.drop(labels=dropped_columns, axis=1)

mean = data.sum(numeric_only=True) / 699
std = data.std(numeric_only=True)
print(data)
data = (data - mean )/ std


     Cart_Position_X  Pendulum_Angle_Rad  Velocity_dX_dT  \
0            -0.0001              0.1012         -0.0035   
1            -0.0004              0.1043         -0.0070   
2            -0.0008              0.1093         -0.0105   
3            -0.0015              0.1163         -0.0140   
4            -0.0023              0.1254         -0.0177   
..               ...                 ...             ...   
694           3.0301             -4.4396          0.0489   
695           3.0328             -4.4664          0.0579   
696           3.0359             -4.4748          0.0646   
697           3.0393             -4.4653          0.0707   
698           3.0431             -4.4383          0.0778   

     AngVelocity_dTheta_dT  Force_Applied  
0                   0.0388            0.0  
1                   0.0771            0.0  
2                   0.1156            0.0  
3                   0.1552            0.0  
4                   0.1964            0.0  
..             

In [19]:
actions_df = data['Force_Applied']
states_df = data.drop(axis=1, labels=['Force_Applied'])

actions = np.array([actions_df])
# actions = np.where((actions > -0.08) & (actions < 0.01), 0, actions)
states = np.array([states_df]).reshape(699, 4)
states = eops.rearrange(states, "s f -> f s")

In [20]:
actions[:, :10]

array([[-0.07679437, -0.07679437, -0.07679437, -0.07679437, -0.07679437,
        -0.07679437, -0.07679437, -0.07679437, -0.07679437, -0.07679437]])

In [22]:
model = NextStateModel(4, 1)
model.train(states, actions, epochs=5000)

Err standing at -0.011723469969037142 at 0th epochs
Err standing at -0.011785265995218386 at 1th epochs
Err standing at -0.011844968007602149 at 2th epochs
Err standing at -0.011902643911139443 at 3th epochs
Err standing at -0.011958359346375349 at 4th epochs
Err standing at -0.012012177766887278 at 5th epochs
Err standing at -0.012064160514014136 at 6th epochs
Err standing at -0.012114366888992884 at 7th epochs
Err standing at -0.012162854222584113 at 8th epochs
Err standing at -0.012209677942269695 at 9th epochs
Err standing at -0.012254891637129877 at 10th epochs
Err standing at -0.012298547120447657 at 11th epochs
Err standing at -0.012340694490158845 at 12th epochs
Err standing at -0.012381382187187614 at 13th epochs
Err standing at -0.012420657051772685 at 14th epochs
Err standing at -0.012458564377833482 at 15th epochs
Err standing at -0.012495147965446166 at 16th epochs
Err standing at -0.01253045017152563 at 17th epochs
Err standing at -0.012564511958722357 at 18th epochs
Err 

In [ ]:
"""------------------------------------------RL PART OF THE ALGORITHM---------------------------------------------------"""

In [87]:
def phi_x(data :np.ndarray):
    disp_vel = data[0]*data[2]
    angle_angVel = data[1]*data[3]
    new_col = np.vstack((disp_vel, angle_angVel))
    data = np.vstack((data,new_col))
    return data

In [124]:
phi_states = phi_x(states)
phi_states

array([[-3.07045925e-01, -3.07189388e-01, -3.07380673e-01, ...,
         1.14480485e+00,  1.14643077e+00,  1.14824798e+00],
       [ 1.25434190e+00,  1.25574122e+00,  1.25799819e+00, ...,
        -8.11237765e-01, -8.06949520e-01, -7.94761878e-01],
       [-2.39697976e-02, -2.49514001e-02, -2.59330026e-02, ...,
        -4.87061800e-03, -3.15982512e-03, -1.16857438e-03],
       [ 6.40867501e-02,  7.87289105e-02,  9.34475312e-02, ...,
         4.00399645e-02,  1.75451275e-01,  3.07307179e-01],
       [ 7.35982868e-03,  7.66480534e-03,  7.97130379e-03, ...,
        -5.57590713e-03, -3.62252075e-03, -1.34181316e-03],
       [ 8.03866957e-02,  9.88631380e-02,  1.17556825e-01, ...,
        -3.24819313e-02, -1.41580322e-01, -2.44236031e-01]])

In [23]:
def reward_fn(state :np.ndarray):
    reward = 0.9*state[:,1]**2-0.5*state[:,3]**2
    return float(-reward)

In [24]:
np.min(actions[:,:])

np.float64(-2.9614972409203593)

In [25]:
class RL():
    def __init__(self, _nstates :int, _nactions :int, rewardFn_):
        self.theta = np.zeros(shape=(1, _nstates))
        self._nState = _nstates
        self._nActions = _nactions
        self.reward_fn = lambda state : rewardFn_(state)

    def runSGD(self, y_i :np.ndarray, _state :np.ndarray):
        diff = 2 * self.theta @ _state - y_i
        grad = diff * _state
        self.theta -= self.lr*grad

    #Follows Fitted value iteration
    def train(self, States :np.ndarray, Actions :np.ndarray, lr :float=1e-2, epochs :int=1000, gamma :float=0.99):
        ns_pred_model = model
        
        if States.ndim != 2 or Actions.ndim != 2:
            print("Dimension mismatch")
            return
            
        _nState, _nSample = States.shape
        _nAction, _ = Actions.shape


        if _nState!=self._nState or _nAction != self._nAction:
            print(f"Req:({self._nState},{self._nAction}) provided:({_nState},{_nAction})")
            return

        theta = self.theta
        self.lr = lr
        
        for iter_ in range(epochs):
            for i in range(_nSample):
                state = States[:, i:i+1]
                
                ns_pred = np.array([model.__predict__(state, action).flatten() for action in np.arange(-3, 3.25, 0.25)])
                rewards = self.reward_fn(state)
                
                print(ns_pred.shape)
                
                q_a = np.array([rewards+gamma*(theta@ns) for ns in ns_pred])
                print(q_a.shape)
                y_i = np.max(q_a)
                
                self.runSGD(y_i, state)
        
                
                                
                
                
                              
            
        
        
        

    

        